<a href="https://colab.research.google.com/github/finneKIM/stem-remix-assistant/blob/main/notebooks/01_setup_and_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01. 환경 세팅 및 생성/분리 단독 테스트

이 노트북은 Colab 무료 T4 GPU에서 실행하는 것을 전제로 한다.

실행 순서: 런타임 -> 런타임 유형 변경 -> T4 GPU 선택 후 아래 셀을 순서대로 실행한다.

목적: MusicGen 단독 생성, Demucs 단독 분리가 각각 정상 동작하는지 확인한다. 재생성/재조합 로직은 이후 노트북에서 다룬다.

## 0. GPU 확인

In [1]:
!nvidia-smi

Fri Aug 14 05:00:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. 라이브러리 설치

AudioCraft(MusicGen)와 Demucs를 설치한다. Colab은 Linux 기반이라 별도 WSL 없이 바로 설치된다.

In [3]:
import sys
!{sys.executable} -m pip install --force-reinstall --no-cache-dir torch torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 174.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 109.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 125.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 107.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 120.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import sys
!{sys.executable} -c "import torch, torchaudio; print(torch.__version__, torchaudio.__version__, torch.cuda.is_available())"

2.13.0+cu130 2.11.0+cu130 True


In [2]:
import sys, os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

!{sys.executable} -m pip install -q --no-deps git+https://github.com/facebookresearch/audiocraft
!{sys.executable} -m pip install -q einops flashy hydra-core hydra_colorlog julius num2words sentencepiece huggingface_hub tqdm "transformers>=4.31.0" librosa soundfile gradio torchmetrics encodec protobuf pesq pystoi torchdiffeq spacy av demucs
!apt-get -qq install -y ffmpeg

!{sys.executable} -c "from audiocraft.models import MusicGen; print('import 성공')"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 3.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 66.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 11.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.8/155.8 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 48.7 MB/s eta 0:00:00
   ━

In [2]:
import sys
!{sys.executable} -c "from audiocraft.models import MusicGen; print('import 성공')"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/usr/local/lib/python3.12/dist-packages/audiocraft/__init__.py", line 26, in <module>
    from . import data, modules, models
  File "/usr/local/lib/python3.12/dist-packages/audiocraft/data/__init__.py", line 10, in <module>
    from . import audio, audio_dataset, info_audio_dataset, music_dataset, sound_dataset, jasco_dataset
  File "/usr/local/lib/python3.12/dist-packages/audiocraft/data/info_audio_dataset.py", line 19, in <module>
    from ..modules.conditioners import SegmentWithAttributes, ConditioningAttributes
  File "/usr/local/lib/python3.12/dist-packages/audiocraft/modules/__init__.py", line 22, in <module>
    from .transformer import StreamingTransformer
  File "/usr/local/lib/python3.12/dist-packages/audiocraft/modules/transformer.py", line 23, in <module>
    from xformers import ops
ModuleNotFoundError: No module named 'xformers'


In [3]:
import sys
!{sys.executable} -m pip install --no-cache-dir xformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 65.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
audiocraft 1.4.0a2 requires torchtext==0.16.0, which is not installed.
audiocraft 1.4.0a2 requires av==11.0.0, but you have av 18.1.0 which is incompatible.
audiocraft 1.4.0a2 requires numpy<2.0.0, but you have numpy 2.5.2 which is incompatible.
audiocraft 1.4.0a2 requires spacy==3.7.6, but you have spacy 3.8.14 which is incompatible.
audiocraft 1.4.0a2 requires torch==2.1.0, but you have torch 2.13.0 which is incompatible.
audiocraft 1.4.0a2 requires torchaudio<2.1.2,>=2.0.0, but you have torchaudio 2.11.0 which is incompatible.
audiocraft 1.4.0a2 requires torchvision==0.16.0, but you have torchvision 0.28.0 which is incompatible.
audiocraft 1.4.0a2 requires xformers<0.0.23, but you have xformers 0.0.35 which is incompatible.


In [4]:
!{sys.executable} -c "from audiocraft.models import MusicGen; print('import 성공')"

2026-08-14 05:14:59.271172: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/usr/local/lib/python3.12/dist-packages/audiocraft/__init__.py", line 26, in <module>
    from . import data, modules, models
  File "/usr/local/lib/python3.12/dist-packages/audiocraft/data/__init__.py", line 10, in <module>
    from . import audio, audio_dataset, info_audio_dataset, music_dataset, sound_dataset, jasco_dataset
  File "/usr/local/lib/python3.12/dist-packages/audiocraft/data/info_audio_dataset.py", line 19, in <module>
    from ..modules.conditioners import SegmentWithAttributes, ConditioningAttributes
  File "/usr/local/lib/python3.12/dist-packages/audiocraft/modules/conditio

In [5]:
import sys
!{sys.executable} -m pip install --no-cache-dir "numpy==1.26.4"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 238.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.2
    Uninstalling numpy-2.5.2:
      Successfully uninstalled numpy-2.5.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
audiocraft 1.4.0a2 requires torchtext==0.16.0, which is not installed.
audiocraft 1.4.0a2 requires av==11.0.0, but you have av 18.1.0 which is incompatible.
audiocraft 1.4.0a2 requires spacy==3.7.6, but you have spacy 3.8.14 which is incompatible.
audiocraft 1.4.0a2 requires torch==2.1.0, but you have torch 2.13.0 which is incompatible.
audiocraft 1.4.0a2 requires torchaudio<2.1.2,>=2.0.0, but you have torchaudio 2.11.0 which is incompatible.
audiocraft 1.4.0a2 requires torchvision==0.16.0, but you have torchvis

In [6]:
!{sys.executable} -c "from audiocraft.models import MusicGen; print('import 성공')"

2026-08-14 05:16:29.729270: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
import 성공


In [9]:
import sys
!{sys.executable} -m pip uninstall -y numpy
!{sys.executable} -m pip install --no-cache-dir --force-reinstall "numpy==1.26.4"

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 280.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
audiocraft 1.4.0a2 requires torchtext==0.16.0, which is not installed.
audiocraft 1.4.0a2 requires av==11.0.0, but you have av 18.1.0 which is incompatible.
audiocraft 1.4.0a2 requires spacy==3.7.6, but you have spacy 3.8.14 which is incompatible.
audiocraft 1.4.0a2 requires torch==2.1.0, but you have torch 2.13.0 which is incompatible.
audiocraft 1.4.0a2 requires torchaudio<2.1.2,>=2.0.0, but you have torchaudio 2.11.0 which is incompatible.
audiocraft 1.4.0a2 requires torchvision==0.16.0, but you have torchvision 0.28.0 which is incompatible.
audio

## 2. Google Drive 마운트 (중간 산출물 저장용)

Colab 무료 티어는 세션이 끊기면 로컬 파일이 사라진다. 생성/분리 결과를 Drive에 저장해 재개 가능하도록 한다.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/stem-remix-assistant/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(OUTPUT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/stem-remix-assistant/outputs


## 3. MusicGen 단독 생성 테스트

medium 모델, stereo, 짧은 길이(15초)로 우선 테스트한다. 문제 없으면 길이를 늘린다.

In [2]:
import torch
from audiocraft.models import MusicGen
from audiocraft.data.audio import audio_write

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

model = MusicGen.get_pretrained('facebook/musicgen-stereo-medium', device=device)
model.set_generation_params(duration=15)

prompt = ["lofi hiphop with mellow piano, 90 BPM, warm and relaxed"]
wav = model.generate(prompt)

for idx, one_wav in enumerate(wav):
    audio_write(f'{OUTPUT_DIR}/draft_{idx}', one_wav.cpu(), model.sample_rate, strategy="loudness")

print('생성 완료')

device: cuda


state_dict.bin: reconstructing file:   0%|          |  0.00B / 3.73GB            

state_dict.bin: downloading bytes:           |  0.00B            

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

... (6 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  236MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/116 [00:00<?, ?it/s]

생성 완료


In [3]:
# 사용량 체크
print('device 변수:', device)
print('모델 파라미터가 실제로 올라간 위치:', next(model.lm.parameters()).device)
!nvidia-smi

device 변수: cuda
모델 파라미터가 실제로 올라간 위치: cuda:0
Fri Aug 14 05:27:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   66C    P0             31W /   70W |    5679MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+---

## 4. Demucs 단독 분리 테스트

방금 생성한 트랙을 4-스템(드럼/베이스/보컬/기타)으로 분리한다.

In [4]:
!demucs -n htdemucs "{OUTPUT_DIR}/draft_0.wav" -o "{OUTPUT_DIR}/separated"

htdemucs.yaml: 100% 21.0/21.0 [00:00<00:00, 104kB/s]

955717e8.safetensors: downloading bytes:  25% 20.7M/84.0M [00:00<00:01, 34.1MB/s, 1.04MB/s  ]
955717e8.safetensors: reconstructing file:   1% 1.02M/84.0M [00:00<01:20, 1.03MB/s]
955717e8.safetensors: reconstructing file:   1% 1.25M/84.0M [00:01<01:09, 1.19MB/s, 99.7kB/s  ]
955717e8.safetensors: downloading bytes:  36% 30.4M/84.0M [00:01<00:01, 33.2MB/s, 2.16MB/s  ]
955717e8.safetensors: downloading bytes:  45% 38.1M/84.0M [00:01<00:01, 40.9MB/s, 2.91MB/s  ]
955717e8.safetensors: downloading bytes:  72% 60.7M/84.0M [00:01<00:00, 59.3MB/s, 4.59MB/s  ]
955717e8.safetensors: downloading bytes:  83% 70.0M/84.0M [00:01<00:00, 54.8MB/s, 6.35MB/s  ]
955717e8.safetensors: downloading bytes:  97% 81.2M/84.0M [00:01<00:00, 55.0MB/s, 7.01MB/s  ]
955717e8.safetensors: downloading bytes: 100% 84.0M/84.0M [00:02<00:00, 36.6MB/s, 7.46MB/s  ]
955717e8.safetensors: reconstructing file: 100% 84.0M/84.0M [00:02<00:00, 36.6MB/s, 7.63MB/s  ]
Selected mod

## 5. 결과 확인

In [5]:
import IPython.display as ipd
import glob

for f in sorted(glob.glob(f'{OUTPUT_DIR}/separated/htdemucs/draft_0/*.wav')):
    print(f)
    display(ipd.Audio(f))

## 6. 체크리스트

- [ ] GPU 정상 인식
- [ ] MusicGen 생성 성공 (에러 없이 wav 생성)
- [ ] Demucs 분리 성공 (4개 스템 파일 생성)
- [ ] Drive에 결과 저장 확인

문제가 생기면 `docs/troubleshooting/`에 날짜별로 기록한다.